# Local Aurora forecast experiments

Load immutable pipeline results. No data acquisition, model inference or training runs here. Synthetic fixtures and development/reference agreement must never be interpreted as qualification.


In [ ]:
import os
from pathlib import Path
from tempfile import TemporaryDirectory

from IPython.display import Image, display

from aurora_finetuning.experiment_reports import (
    example_plots,
    execution_results,
    legacy_metrics,
    load_results,
    paired_summary,
    performance_plots,
)

REPO = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()
)
RESULTS = Path(
    os.environ.get("AURORA_RESULTS_ROOT", str(REPO / "outputs/experiments"))
).resolve()
results = load_results(RESULTS)
print(f"Result root: {RESULTS}")
print(f"Loaded {len(results)} score rows")
if results.empty:
    print(
        "Run the README smoke example, or set AURORA_RESULTS_ROOT "
        "to completed campaign outputs."
    )

## Coverage and study population

Inspect track, split, event counts and available methods before comparing errors. The latest snapshot is used once per run. Lead 0 denotes a complete-horizon rainfall total.


In [ ]:
if not results.empty:
    display(
        results.groupby(
            ["campaign", "track", "split", "method", "variable", "interval_hours"]
        ).agg(events=("case", "nunique"), score_rows=("metric", "size"))
    )
    display(results[(results.metric == "coverage") & (results.baseline == "own")])

## Paired performance and uncertainty

Negative paired MAE differences improve on the named baseline. Intervals bootstrap independent events; one event receives no confidence interval. Own-support curves can differ in coverage and must not be ranked directly.


In [ ]:
paired = paired_summary(results, seed=42)
display(paired)
with TemporaryDirectory(prefix="aurora-plots-") as directory:
    for path in performance_plots(results, Path(directory)):
        display(Image(filename=str(path)))

## Event examples and rainfall structure

The first stored location is shown without spatial averaging. Native rain maps appear only when saved grids exist. Check forecast interval and model age in the saved manifest before interpreting differences.


In [ ]:
with TemporaryDirectory(prefix="aurora-examples-") as directory:
    for path in example_plots(RESULTS, Path(directory)):
        display(Image(filename=str(path)))

## Runtime, missing inputs and failures

Worker time includes process startup. Model timings, RSS and MPS allocation remain separate in NetCDF execution metadata and original job reports. Reused outputs are not new timing measurements.


In [ ]:
execution = execution_results(RESULTS)
display(execution)
if not execution.empty and "seconds" in execution:
    measured = execution[execution["reused"].fillna(False).eq(False)].dropna(
        subset=["seconds"]
    )
    if not measured.empty:
        ax = measured.plot.barh(
            x="unit",
            y="seconds",
            figsize=(10, max(3, len(measured) * 0.2)),
            legend=False,
        )
        ax.set_xlabel("Worker wall time (s)")

## Historical evidence

The tracked experiment history and case reports document previous tests. Use `pixi run experiment-collect --output outputs/evidence-new` to inventory local artifacts; its development comparison CSV is separate from observational scores. Missing original artifacts cannot establish reproducibility.


In [ ]:
legacy = legacy_metrics(REPO)
print(f"Loaded {len(legacy)} original historical metric records")
if not legacy.empty:
    display(legacy.groupby(["artifact", "scope"]).size().rename("metric_records"))
    display(legacy)
# Filter by artifact and full metric_path to inspect a particular method/lead.
# These values have incompatible historical support definitions: do not pool them.

## Interpretation checklist

State which variable, horizon, season, region and weather regime improved; compare shared support and availability assumptions. Report negative findings, uncertainty, resource cost, and archive limitations. Synthetic results demonstrate software behaviour only.
